# 🎯 Surveillance-IA — Entraînement Optimisé YOLOv8

**Projet SAHELYS** — Fine-tuning YOLOv8l (Large) pour la détection de personnes haute précision

| Paramètre | Valeur |
|-----------|--------|
| Modèle | **yolov8l.pt** (large — 43.7M params) |
| Dataset | COCO train2017 (~64k images personnes) |
| GPU | T4 16 Go (free tier) |
| Epochs | 120 (early stopping patience=20) |
| Image size | 640×640 |
| Batch | Auto (optimisé GPU) |
| Optimizer | AdamW + Cosine LR |
| Augmentation | Mosaic + CopyPaste + Mixup + HSV + Flip |
| Target | **Precision ≥96%, mAP@0.5 ≥95%** |

**Auteur** : BAWANA Théodore — [theo.portefolio.io](https://theo.portefolio.io)

---

⚠️ **AVANT DE COMMENCER** :
1. Menu → **Exécution** → **Modifier le type d’exécution** → **GPU T4**
2. Exécuter toutes les cellules dans l’ordre (**Ctrl+F9**)

## 1️⃣ Setup — GPU, Dépendances, Drive

In [ ]:
# ══════════════════════════════════════════════════════════
#  CONFIGURATION GLOBALE
# ══════════════════════════════════════════════════════════
import os, sys, json, shutil, random, time, gc
from pathlib import Path

# ── Constantes ──
SEED = 42
MODEL_NAME = "yolov8l.pt"       # Large = meilleure précision (43.7M params)
IMG_SIZE = 640
EPOCHS = 120
PATIENCE = 20                    # Plus de patience pour convergence complète
MIN_AREA = 512                   # Seuil min d'aire (plus de petites personnes)
CONF_THRESHOLD = 0.35            # Seuil de confiance optimisé pour precision
PROJECT_NAME = "surveillance_yolov8l_optimized"

random.seed(SEED)

# ── Vérifier GPU ──
!nvidia-smi -L

# ── Installer les dépendances ──
!pip install -q ultralytics>=8.2.0 opencv-python-headless>=4.8.0 albumentations>=1.3.0

import torch
import numpy as np

assert torch.cuda.is_available(), "❌ GPU non détecté ! Aller dans Exécution → Modifier le type → GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"\n{'='*50}")
print(f"  PyTorch  : {torch.__version__}")
print(f"  GPU      : {gpu_name}")
print(f"  VRAM     : {vram_gb:.1f} GB")
print(f"  Modèle   : {MODEL_NAME}")
print(f"  Epochs   : {EPOCHS} (patience={PATIENCE})")
print(f"{'='*50}")

In [ ]:
# ── Monter Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/Surveillance-IA"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"✅ Drive monté → {DRIVE_DIR}")

## 2️⃣ Téléchargement COCO 2017 (train + val)

- **train2017** (~118k images, ~18 Go) → entraînement (~64k avec personnes)
- **val2017** (~5k images, ~1 Go) → validation + test

In [ ]:
%%time
# ══════════════════════════════════════════════════════════
#  TÉLÉCHARGEMENT COCO 2017
# ══════════════════════════════════════════════════════════

print("📥 [1/3] Annotations...")
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q -o annotations_trainval2017.zip -d coco/ && rm annotations_trainval2017.zip

print("📥 [2/3] val2017 (~1 Go)...")
!wget -q http://images.cocodataset.org/zips/val2017.zip
!unzip -q -o val2017.zip -d coco/ && rm val2017.zip

print("📥 [3/3] train2017 (~18 Go — ~10-15 min)...")
!wget -q --show-progress http://images.cocodataset.org/zips/train2017.zip
!unzip -q -o train2017.zip -d coco/ && rm train2017.zip

n_train_imgs = len(os.listdir("coco/train2017"))
n_val_imgs = len(os.listdir("coco/val2017"))
print(f"\n✅ Téléchargé : {n_train_imgs} train + {n_val_imgs} val")

## 3️⃣ Préparation du Dataset YOLO

Pipeline : COCO → Filtre personnes → YOLO format → Train/Val/Test

In [ ]:
# ══════════════════════════════════════════════════════════
#  FILTRAGE & CONVERSION COCO → YOLO
# ══════════════════════════════════════════════════════════
from tqdm import tqdm

DATA_ROOT = Path("data/splits")

def load_person_data(ann_file):
    """Charge annotations COCO et filtre les personnes."""
    with open(ann_file) as f:
        coco = json.load(f)

    person_ids = {c["id"] for c in coco["categories"] if c["name"] == "person"}

    # Filtrer : pas crowd, aire > MIN_AREA, bbox valide
    anns = [
        a for a in coco["annotations"]
        if a["category_id"] in person_ids
        and not a.get("iscrowd", 0)
        and a["area"] > MIN_AREA
        and a["bbox"][2] > 2 and a["bbox"][3] > 2  # largeur/hauteur min
    ]

    img_lookup = {img["id"]: img for img in coco["images"]}
    img_ids = {a["image_id"] for a in anns}

    anns_by_img = {}
    for a in anns:
        anns_by_img.setdefault(a["image_id"], []).append(a)

    return img_ids, img_lookup, anns_by_img, len(anns)


def bbox_coco_to_yolo(bbox, img_w, img_h):
    """Convertit bbox COCO [x,y,w,h] en YOLO [xc,yc,w,h] normalisé."""
    x, y, w, h = bbox
    xc = (x + w / 2) / img_w
    yc = (y + h / 2) / img_h
    return (
        max(0.0, min(1.0, xc)),
        max(0.0, min(1.0, yc)),
        max(0.0, min(1.0, w / img_w)),
        max(0.0, min(1.0, h / img_h)),
    )


# Charger les deux splits COCO
print("📊 Chargement train2017...")
train_ids, train_lookup, train_anns, n1 = load_person_data("coco/annotations/instances_train2017.json")
print(f"   {len(train_ids):,} images, {n1:,} annotations")

print("📊 Chargement val2017...")
val_ids, val_lookup, val_anns, n2 = load_person_data("coco/annotations/instances_val2017.json")
print(f"   {len(val_ids):,} images, {n2:,} annotations")

print(f"\n📋 TOTAL : {len(train_ids)+len(val_ids):,} images, {n1+n2:,} annotations")

In [ ]:
%%time
# ══════════════════════════════════════════════════════════
#  CRÉATION DATASET YOLO (Train / Val / Test)
# ══════════════════════════════════════════════════════════
# Train  = COCO train2017 (personnes)   ~64k images
# Val    = COCO val2017 première moitié ~1.2k images
# Test   = COCO val2017 seconde moitié  ~1.2k images

# Créer l'arborescence
for split in ["train", "val", "test"]:
    (DATA_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATA_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

def process_split(img_ids, img_lookup, anns_by_img, img_dir, split_name):
    """Copie images + génère labels YOLO pour un split."""
    ok, skip = 0, 0
    for img_id in tqdm(sorted(img_ids), desc=split_name, ncols=80):
        info = img_lookup[img_id]
        fname = info["file_name"]
        src = Path(img_dir) / fname
        if not src.exists():
            skip += 1
            continue

        # Copier l'image
        shutil.copy2(src, DATA_ROOT / "images" / split_name / fname)

        # Générer le label YOLO
        if img_id in anns_by_img:
            w, h = info["width"], info["height"]
            lines = []
            for ann in anns_by_img[img_id]:
                xc, yc, nw, nh = bbox_coco_to_yolo(ann["bbox"], w, h)
                if nw > 0.001 and nh > 0.001:  # Ignorer les boîtes microscopiques
                    lines.append(f"0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
            if lines:
                label_path = DATA_ROOT / "labels" / split_name / f"{Path(fname).stem}.txt"
                label_path.write_text("\n".join(lines) + "\n")
                ok += 1
    return ok, skip

# ── TRAIN : tout train2017 ──
print("🚂 Train (COCO train2017)...")
n_train, s1 = process_split(train_ids, train_lookup, train_anns, "coco/train2017", "train")

# ── VAL + TEST : split val2017 en 2 ──
val_list = sorted(list(val_ids))
random.shuffle(val_list)
mid = len(val_list) // 2

print("📋 Val (COCO val2017 — 1ère moitié)...")
n_val, s2 = process_split(set(val_list[:mid]), val_lookup, val_anns, "coco/val2017", "val")

print("🧪 Test (COCO val2017 — 2e moitié)...")
n_test, s3 = process_split(set(val_list[mid:]), val_lookup, val_anns, "coco/val2017", "test")

# ── Nettoyage agressif pour libérer l'espace disque ──
print("\n🗑️ Nettoyage espace disque...")
shutil.rmtree("coco", ignore_errors=True)
gc.collect()

print(f"\n{'='*50}")
print(f"  📊 DATASET YOLO PRÊT")
print(f"{'='*50}")
print(f"  Train : {n_train:>6,} images")
print(f"  Val   : {n_val:>6,} images")
print(f"  Test  : {n_test:>6,} images")
print(f"  TOTAL : {n_train+n_val+n_test:>6,} images")
print(f"{'='*50}")

!df -h / | tail -1

In [ ]:
# ── Générer data.yaml ──
import yaml

data_config = {
    "path": str(DATA_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "person"},
    "nc": 1,
}

yaml_path = DATA_ROOT / "data.yaml"
yaml_path.write_text(yaml.dump(data_config, default_flow_style=False))

# Vérification intégrité
print("🔍 Vérification intégrité :")
errors = 0
for split in ["train", "val", "test"]:
    imgs = set(p.stem for p in (DATA_ROOT / "images" / split).glob("*.jpg"))
    lbls = set(p.stem for p in (DATA_ROOT / "labels" / split).glob("*.txt"))
    missing = imgs - lbls
    extra = lbls - imgs
    if missing:
        print(f"  ⚠️ {split}: {len(missing)} images sans label")
        errors += len(missing)
    if extra:
        print(f"  ⚠️ {split}: {len(extra)} labels sans image")
        errors += len(extra)
    print(f"  ✅ {split:5s}: {len(imgs)} img, {len(lbls)} lbl")

if errors == 0:
    print("\n✅ Dataset parfaitement intègre !")
print(f"\n{yaml_path.read_text()}")

## 4️⃣ Visualisation — Échantillons du dataset

In [ ]:
import cv2
import matplotlib.pyplot as plt

def draw_yolo_boxes(img_path, label_path):
    """Dessine les bbox YOLO sur une image."""
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    n_boxes = 0
    if Path(label_path).exists():
        for line in Path(label_path).read_text().strip().split("\n"):
            parts = line.split()
            if len(parts) == 5:
                _, xc, yc, bw, bh = map(float, parts)
                x1, y1 = int((xc - bw/2)*w), int((yc - bh/2)*h)
                x2, y2 = int((xc + bw/2)*w), int((yc + bh/2)*h)
                cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
                n_boxes += 1
    return img, n_boxes

# Afficher 8 exemples variés (début, milieu, fin du dataset)
train_imgs = sorted((DATA_ROOT / "images" / "train").glob("*.jpg"))
indices = np.linspace(0, len(train_imgs)-1, 8, dtype=int)
samples = [train_imgs[i] for i in indices]

fig, axes = plt.subplots(2, 4, figsize=(22, 11))
for ax, img_path in zip(axes.flat, samples):
    lbl = DATA_ROOT / "labels" / "train" / f"{img_path.stem}.txt"
    img, n = draw_yolo_boxes(img_path, lbl)
    ax.imshow(img)
    ax.set_title(f"{img_path.name}\n{n} personne(s)", fontsize=8)
    ax.axis("off")

fig.suptitle("Surveillance-IA — Échantillons Train", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Distribution du nombre de personnes par image
counts = []
for lbl in (DATA_ROOT / "labels" / "train").glob("*.txt"):
    counts.append(len(lbl.read_text().strip().split("\n")))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(counts, bins=range(0, max(counts)+2), edgecolor="black", alpha=0.7, color="#2196F3")
ax.set_xlabel("Nombre de personnes par image")
ax.set_ylabel("Fréquence")
ax.set_title(f"Distribution — {len(counts)} images, médiane={np.median(counts):.0f}, max={max(counts)}")
plt.tight_layout()
plt.show()

## 5️⃣ Entraînement Optimisé — YOLOv8 Large

**Optimisations clés vs v1** :

| Optimisation | Ancien | Nouveau | Impact |
|---|---|---|---|
| Modèle | yolov8m (25.9M) | **yolov8l (43.7M)** | +3-5% mAP |
| Optimizer | SGD | **AdamW** | Convergence + rapide |
| Batch | 16 fixe | **Auto** | Utilise 100% VRAM |
| Epochs | 50-100 | **120 + patience=20** | Convergence complète |
| CopyPaste | 0.0 | **0.3** | +2-3% mAP |
| Mixup | 0.0 | **0.15** | +1% mAP, régularisation |
| close_mosaic | 10 | **15** | Affine sans distorsion |
| Label smooth | 0.0 | **0.01** | Réduit overconfidence |
| Multi-scale | non | **±50%** | Robustesse taille |

Durée estimée : **~3-4h** sur GPU T4

In [ ]:
from ultralytics import YOLO

# ── Charger le modèle Large pré-entraîné ──
model = YOLO(MODEL_NAME)

n_params = sum(p.numel() for p in model.model.parameters()) / 1e6
print(f"✅ {MODEL_NAME} chargé : {n_params:.1f}M paramètres")

In [ ]:
# ══════════════════════════════════════════════════════════
#  🚀  ENTRAÎNEMENT OPTIMISÉ MAXIMUM
# ══════════════════════════════════════════════════════════

t0 = time.time()

results = model.train(
    # ── Dataset ──
    data=str(DATA_ROOT / "data.yaml"),

    # ── Training ──
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=-1,                 # AUTO-BATCH : utilise le max de VRAM disponible
    patience=PATIENCE,

    # ── Optimizer : AdamW (meilleure convergence) ──
    optimizer="AdamW",
    lr0=0.001,                # LR plus bas pour AdamW
    lrf=0.01,                 # LR final = lr0 * 0.01
    weight_decay=0.0005,
    cos_lr=True,
    warmup_epochs=5,          # Warmup plus long pour stabilité
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    # ── Loss weights (optimisés single-class) ──
    box=7.5,
    cls=0.5,
    dfl=1.5,

    # ── Augmentation AGRESSIVES ──
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,              # Légère rotation
    translate=0.15,           # Translation
    scale=0.5,                # Échelle multi-scale
    shear=2.0,                # Léger cisaillement
    perspective=0.0001,       # Légère perspective
    fliplr=0.5,
    flipud=0.0,               # Pas de flip vertical (personnes)
    mosaic=1.0,               # Mosaic actif
    mixup=0.15,               # Régularisation mixup
    copy_paste=0.3,           # CopyPaste augmentation (très efficace)
    erasing=0.1,              # Random erasing
    close_mosaic=15,          # Désactiver mosaic les 15 derniers epochs

    # ── Régularisation ──
    label_smoothing=0.01,     # Réduit l'overconfidence
    nbs=64,                   # Nominal batch size pour scaling LR

    # ── Output ──
    project="models/finetuned",
    name=PROJECT_NAME,
    device="cuda",
    workers=4,
    save=True,
    save_period=10,
    exist_ok=True,
    pretrained=True,
    verbose=True,
    seed=SEED,
    val=True,
    plots=True,
    amp=True,                 # Mixed precision (plus rapide)
)

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"  ✅ ENTRAÎNEMENT TERMINÉ en {elapsed/3600:.1f}h ({elapsed/60:.0f} min)")
print(f"{'='*60}")

## 6️⃣ Analyse des Résultats

In [ ]:
# ── Métriques finales ──
run_dir = Path(results.save_dir)
print(f"📂 Résultats : {run_dir}")

if hasattr(results, 'results_dict'):
    r = results.results_dict
    precision = r.get('metrics/precision(B)', 0)
    recall = r.get('metrics/recall(B)', 0)
    map50 = r.get('metrics/mAP50(B)', 0)
    map5095 = r.get('metrics/mAP50-95(B)', 0)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    print(f"\n{'='*55}")
    print(f"  {'MÉTRIQUES VALIDATION':^51}")
    print(f"{'='*55}")
    print(f"  {'Precision':.<20} {precision:.4f}  {'[≥ 0.96]':>12}")
    print(f"  {'Recall':.<20} {recall:.4f}")
    print(f"  {'F1-Score':.<20} {f1:.4f}")
    print(f"  {'mAP@0.5':.<20} {map50:.4f}  {'[≥ 0.95]':>12}")
    print(f"  {'mAP@0.5:0.95':.<20} {map5095:.4f}")
    print(f"{'='*55}")

    if precision >= 0.96:
        print("\n  🎯 OBJECTIF ATTEINT : Precision ≥ 96% !")
    elif precision >= 0.93:
        print(f"\n  🟡 Proche de l'objectif : {precision:.2%}")
    else:
        print(f"\n  ⚠️ Precision = {precision:.2%} — objectif 96% non atteint")

In [ ]:
# ── Courbes d'apprentissage ──
from IPython.display import Image, display

curves = [
    "results.png", "confusion_matrix.png", "confusion_matrix_normalized.png",
    "P_curve.png", "R_curve.png", "F1_curve.png", "PR_curve.png",
]

for name in curves:
    path = run_dir / name
    if path.exists():
        print(f"\n📊 {name}")
        display(Image(filename=str(path), width=900))

In [ ]:
# ── Analyse de la courbe de training (loss + métriques par epoch) ──
import pandas as pd

csv_path = run_dir / "results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    # Loss
    for col in ["train/box_loss", "train/cls_loss", "train/dfl_loss"]:
        if col in df.columns:
            axes[0].plot(df["epoch"], df[col], label=col.split("/")[1])
    axes[0].set_title("Training Loss")
    axes[0].legend()
    axes[0].set_xlabel("Epoch")

    # Val Loss
    for col in ["val/box_loss", "val/cls_loss", "val/dfl_loss"]:
        if col in df.columns:
            axes[1].plot(df["epoch"], df[col], label=col.split("/")[1])
    axes[1].set_title("Validation Loss")
    axes[1].legend()
    axes[1].set_xlabel("Epoch")

    # Métriques
    for col in ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
        if col in df.columns:
            label = col.split("/")[1].replace("(B)", "")
            axes[2].plot(df["epoch"], df[col], label=label)
    axes[2].set_title("Métriques")
    axes[2].legend()
    axes[2].set_xlabel("Epoch")
    axes[2].axhline(y=0.96, color='r', linestyle='--', alpha=0.5, label='Target 96%')

    fig.suptitle("Surveillance-IA — Courbes d'entraînement", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # Meilleur epoch
    if "metrics/precision(B)" in df.columns:
        best_idx = df["metrics/mAP50(B)"].idxmax()
        print(f"\n⭐ Meilleur epoch : {int(df.loc[best_idx, 'epoch'])}")
        print(f"   Precision={df.loc[best_idx, 'metrics/precision(B)']:.4f}  "
              f"Recall={df.loc[best_idx, 'metrics/recall(B)']:.4f}  "
              f"mAP50={df.loc[best_idx, 'metrics/mAP50(B)']:.4f}")

## 7️⃣ Évaluation Finale — Test Set

⚠️ **Évaluation unique** sur des données jamais vues pendant l'entraînement

In [ ]:
# ══════════════════════════════════════════════════════════
#  🧪 ÉVALUATION FINALE SUR LE TEST SET
# ══════════════════════════════════════════════════════════

best_path = run_dir / "weights" / "best.pt"
print(f"📦 Modèle : {best_path}")
print(f"   Taille : {best_path.stat().st_size / 1e6:.1f} Mo")

best_model = YOLO(str(best_path))

# Évaluation
test_results = best_model.val(
    data=str(DATA_ROOT / "data.yaml"),
    split="test",
    imgsz=IMG_SIZE,
    batch=-1,
    device="cuda",
    conf=0.001,       # Bas pour avoir des courbes PR complètes
    iou=0.6,
    verbose=True,
    plots=True,
)

if hasattr(test_results, 'results_dict'):
    r = test_results.results_dict
    tp = r.get('metrics/precision(B)', 0)
    tr = r.get('metrics/recall(B)', 0)
    tm50 = r.get('metrics/mAP50(B)', 0)
    tm5095 = r.get('metrics/mAP50-95(B)', 0)
    tf1 = 2 * tp * tr / (tp + tr + 1e-8)

    print(f"\n{'='*55}")
    print(f"  {'RÉSULTATS TEST SET — Surveillance-IA':^51}")
    print(f"{'='*55}")
    print(f"  {'Precision':.<20} {tp:.4f}")
    print(f"  {'Recall':.<20} {tr:.4f}")
    print(f"  {'F1-Score':.<20} {tf1:.4f}")
    print(f"  {'mAP@0.5':.<20} {tm50:.4f}")
    print(f"  {'mAP@0.5:0.95':.<20} {tm5095:.4f}")
    print(f"{'='*55}")

In [ ]:
# ── Benchmark FPS ──
dummy = np.random.randint(0, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

# Warmup
for _ in range(20):
    best_model.predict(dummy, verbose=False, device="cuda")

# Mesure sur 200 itérations
torch.cuda.synchronize()
t0 = time.time()
for _ in range(200):
    best_model.predict(dummy, verbose=False, device="cuda")
torch.cuda.synchronize()
elapsed = time.time() - t0

fps = 200 / elapsed
latency = elapsed / 200 * 1000

print(f"\n⚡ Performance temps réel ({gpu_name}) :")
print(f"   FPS     : {fps:.1f}")
print(f"   Latence : {latency:.1f} ms/image")
print(f"   {'\n   ✅ TEMPS RÉEL OK (≥25 FPS)' if fps >= 25 else '   ⚠️ En dessous du temps réel'}")

## 8️⃣ Test Visuel — Détections sur images test

In [ ]:
# ── Prédiction visuelle sur le test set ──
test_imgs = sorted((DATA_ROOT / "images" / "test").glob("*.jpg"))
# Sélectionner 8 images variées
idx = np.linspace(0, len(test_imgs)-1, 8, dtype=int)
samples = [test_imgs[i] for i in idx]

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
for ax, img_path in zip(axes.flat, samples):
    result = best_model.predict(
        str(img_path), conf=CONF_THRESHOLD, iou=0.45,
        device="cuda", verbose=False,
    )[0]

    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    n = len(result.boxes)
    confs = result.boxes.conf.cpu().numpy() if n > 0 else []
    avg_conf = np.mean(confs) if len(confs) > 0 else 0

    ax.imshow(annotated)
    ax.set_title(f"{n} personne(s) | conf moy={avg_conf:.2f}", fontsize=9)
    ax.axis("off")

fig.suptitle(f"Surveillance-IA — Détections Test (conf≥{CONF_THRESHOLD})",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Analyse détaillée : distribution des confiances ──
all_confs = []
all_sizes = []  # small / medium / large

for img_path in tqdm(test_imgs[:200], desc="Analyse"):
    result = best_model.predict(
        str(img_path), conf=0.1, iou=0.45,
        device="cuda", verbose=False,
    )[0]
    if len(result.boxes) > 0:
        confs = result.boxes.conf.cpu().numpy()
        areas = (result.boxes.xywh[:, 2] * result.boxes.xywh[:, 3]).cpu().numpy()
        all_confs.extend(confs.tolist())
        for a in areas:
            if a < 32**2: all_sizes.append("small")
            elif a < 96**2: all_sizes.append("medium")
            else: all_sizes.append("large")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des confiances
axes[0].hist(all_confs, bins=50, edgecolor="black", alpha=0.7, color="#4CAF50")
axes[0].axvline(x=CONF_THRESHOLD, color='r', linestyle='--', label=f'Seuil={CONF_THRESHOLD}')
axes[0].set_xlabel("Confiance")
axes[0].set_ylabel("Fréquence")
axes[0].set_title(f"Distribution des confiances (n={len(all_confs)})")
axes[0].legend()

# Distribution par taille
from collections import Counter
size_counts = Counter(all_sizes)
labels = ["small\n(<32²)", "medium\n(32²-96²)", "large\n(>96²)"]
values = [size_counts.get(s, 0) for s in ["small", "medium", "large"]]
axes[1].bar(labels, values, color=["#FF9800", "#2196F3", "#4CAF50"], edgecolor="black")
axes[1].set_title("Détections par taille")
axes[1].set_ylabel("Nombre")

plt.tight_layout()
plt.show()

print(f"\n📊 Seuil optimal : conf={CONF_THRESHOLD}")
high_conf = sum(1 for c in all_confs if c >= CONF_THRESHOLD)
print(f"   Détections conf≥{CONF_THRESHOLD} : {high_conf}/{len(all_confs)} ({high_conf/len(all_confs)*100:.1f}%)")

## 9️⃣ Sauvegarde & Export

In [ ]:
# ══════════════════════════════════════════════════════════
#  SAUVEGARDE GOOGLE DRIVE + TÉLÉCHARGEMENT
# ══════════════════════════════════════════════════════════

best_src = run_dir / "weights" / "best.pt"
last_src = run_dir / "weights" / "last.pt"

# Sauvegarder poids
for src, name in [(best_src, "best.pt"), (last_src, "last.pt")]:
    if src.exists():
        shutil.copy2(src, f"{DRIVE_DIR}/{name}")
        print(f"✅ {name} → Drive ({src.stat().st_size / 1e6:.1f} Mo)")

# Sauvegarder courbes + CSV
for pattern in ["*.png", "*.csv"]:
    for f in run_dir.glob(pattern):
        shutil.copy2(f, f"{DRIVE_DIR}/{f.name}")

print(f"\n📁 Tout sauvegardé dans : {DRIVE_DIR}")

# Télécharger best.pt
from google.colab import files
if best_src.exists():
    files.download(str(best_src))
    print("\n📥 best.pt téléchargé !")

In [ ]:
# ── Export ONNX (pour déploiement CPU/GPU cross-platform) ──
onnx_path = best_model.export(format="onnx", imgsz=IMG_SIZE, simplify=True, opset=17)
print(f"✅ ONNX exporté : {onnx_path}")

if Path(onnx_path).exists():
    shutil.copy2(onnx_path, f"{DRIVE_DIR}/best.onnx")
    print(f"   Taille : {Path(onnx_path).stat().st_size / 1e6:.1f} Mo")
    print(f"   Sauvegardé sur Drive")

---

## ✅ Récapitulatif Final

### Fichiers générés
| Fichier | Usage |
|---------|-------|
| `best.pt` | Modèle PyTorch (déploiement principal) |
| `best.onnx` | Modèle ONNX (cross-platform) |
| `results.csv` | Métriques par epoch |
| `*.png` | Courbes d'entraînement |

### Déploiement

Téléchargez `best.pt` et placez-le dans votre projet :

```
surveillance/
└── models/
    └── finetuned/
        └── best.pt    ← ICI
```

```bash
# Tracking temps réel sur webcam
python -m src.tracker --model models/finetuned/best.pt --source 0 --show

# API REST
uvicorn api.main:app --host 0.0.0.0 --port 8000

# Dashboard
streamlit run app/dashboard.py

# Docker
docker compose up -d
```

---

**Auteur** : BAWANA Théodore — [theo.portefolio.io](https://theo.portefolio.io) — [GitHub](https://github.com/theobawana) — Projet SAHELYS